In [ ]:
# Cell 1
import os
import warnings
warnings.filterwarnings('ignore')

# Disable wandb and tokenizer warnings
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Setting up local MacBook M2 environment")
print("Apple Silicon GPU (MPS) optimization enabled")
print("Disabled wandb and tokenizer warnings")


✅ Setting up local MacBook M2 environment
📱 Apple Silicon GPU (MPS) optimization enabled
🚫 Disabled wandb and tokenizer warnings


In [ ]:
# Cell 2: Core Imports
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import time

def get_device():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")
    elif torch.cuda.is_available():
        device = torch.device("cuda") 
        print("Using NVIDIA GPU")
    else:
        device = torch.device("cpu")
        print("Using CPU (slower)")
    return device

device = get_device()
print(f"Device: {device}")

✅ Using Apple Silicon GPU (MPS)
Device: mps


In [ ]:
# Cell 3: Day 2 - Full Parameter Fine-Tuning Setup (Local)
print("===Full Parameter Fine-Tuning Implementation ===")
print("Method 1 of 3: Traditional Full Fine-Tuning")
print("Target: BERT-base + SST-2 dataset")
print(f"Device: {device}")
print("Starting full parameter fine-tuning implementation on MacBook M2...")

=== Day 2: Full Parameter Fine-Tuning Implementation ===
Method 1 of 3: Traditional Full Fine-Tuning
Target: BERT-base + SST-2 dataset
Device: mps
🚀 Starting full parameter fine-tuning implementation on MacBook M2...


In [ ]:
# Cell 4: Task 1 - SST-2 Dataset Loading and Preprocessing
print("=== Loading SST-2 Dataset ===")

# Loading the SST-2 Dataset
from datasets import load_dataset
dataset = load_dataset("glue", 'sst2')

# Display dataset structure
print("\nDataset Structure:")
print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print(f"Testing samples: {len(dataset['test'])}")

# Data Example
print("\nSample Data:")
print("Training example:", dataset['train'][43])
print("Validation example:", dataset['validation'][43])

# Label Example
print("\nLabel Information:")
print("Labels: 0 = Negative, 1 = Positive")
train_labels = dataset['train']['label'][:10000]
print(f"Label distribution: {train_labels.count(0)} negative, {train_labels.count(1)} positive")

=== Loading SST-2 Dataset ===

📊 Dataset Structure:
Training samples: 67349
Validation samples: 872
Testing samples: 1821

🔍 Sample Data:
Training example: {'sentence': 'covers this territory with wit and originality , suggesting that with his fourth feature ', 'label': 1, 'idx': 43}
Validation example: {'sentence': 'holm ... embodies the character with an effortlessly regal charisma . ', 'label': 1, 'idx': 43}

🏷️ Label Information:
Labels: 0 = Negative, 1 = Positive
Label distribution: 4471 negative, 5529 positive


In [ ]:
# Cell 5: Complete Preprocessing Pipeline (Local Optimized)
print("=== Preprocessing Pipeline ===")

# Import and create tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print("Tokenizer loaded")

def preprocess_sst2(examples):
    """
    Preprocessing function for SST-2 dataset
    - Tokenizes text using BERT tokenizer
    - Handles padding and truncation
    - Returns format ready for training
    """
    return tokenizer(
        examples['sentence'],
        truncation=True,
        padding=False,
        max_length=128,
        return_tensors=None
    )

# Use subset for local testing (good for M2 memory)
print("Using subset for local experimentation...")

# Subset for faster local training
train_dataset = dataset['train'].select(range(10000)).map(preprocess_sst2, batched=True)
val_dataset = dataset['validation'].map(preprocess_sst2, batched=True)

print(f"Train dataset: {len(train_dataset)} samples (subset)")
print(f"Validation dataset: {len(val_dataset)} samples (subset)")
print(f"Train dataset columns: {train_dataset.column_names}")
print(f"Sample preprocessed data keys: {list(train_dataset[0].keys())}")

=== Preprocessing Pipeline ===
✅ Tokenizer loaded
Using subset for local experimentation...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

✅ Train dataset: 10000 samples (subset)
✅ Validation dataset: 872 samples (subset)
Train dataset columns: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']
Sample preprocessed data keys: ['sentence', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# Cell 6: BERT Model Setup (Local)
print("=== BERT Model Setup ===")

from transformers import AutoModelForSequenceClassification

model_name = "bert-base-uncased"
print(f"Loading {model_name}...")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

# Check model size and parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Statistics:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params/1_000_000:.1f}M parameters")

# Move model to device (MPS/GPU/CPU)
print(f"Moving model to {device}...")
model = model.to(device)

print("Model setup complete - ready for training!")
print("Optimized for Apple Silicon!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


=== BERT Model Setup ===
Loading bert-base-uncased...
📊 Model Statistics:
Total parameters: 109,483,778
Trainable parameters: 109,483,778
Model size: ~109.5M parameters
Moving model to mps...
✅ Model setup complete - ready for training!
🍎 Optimized for Apple Silicon!


In [ ]:
# Cell 7: Training Arguments (Local Optimized)
print("=== Training Configuration ===")

training_args = TrainingArguments(
    output_dir="./bert-sst2-results",
    num_train_epochs=3,
    per_device_train_batch_size=8,      # Smaller for M2 memory
    per_device_eval_batch_size=8,       # Smaller for M2 memory
    warmup_steps=100,                   # Reduced for smaller dataset
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,                   # More frequent logging
    eval_strategy="steps",
    eval_steps=100,                     # More frequent evaluation
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    dataloader_num_workers=0,           # Stable for M2
    remove_unused_columns=True,
    report_to=None,
    fp16=False,                         # MPS doesn't support fp16 yet
)

print("Training arguments configured for local training")
print(f"Batch size: {training_args.per_device_train_batch_size} (optimized for M2)")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Warmup steps: {training_args.warmup_steps}")
print("Ready for training!")

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


=== Training Configuration ===
✅ Training arguments configured for local training
Batch size: 8 (optimized for M2)
Epochs: 3
Warmup steps: 100
🚀 Ready for training!


In [ ]:
# Cell 8A: Define Evaluation Metrics
print("=== Setting Up Evaluation Metrics ===")

from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    """Compute accuracy for evaluation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

print("Metrics function defined")
print("Ready to create trainer")



=== Setting Up Evaluation Metrics ===
✅ Metrics function defined
✅ Ready to create trainer


In [ ]:
# Cell 8B: Baseline Performance Evaluation (Local)
print("=== Task 4: Baseline Performance Evaluation ===")

from transformers import Trainer, DataCollatorWithPadding
import time


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# Create trainer for evaluation
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

print("Testing UNTRAINED model performance...")
print(f"Evaluating on {len(val_dataset)} validation samples...")
print("This should be fast on local device...")

start_time = time.time()

# Evaluate untrained model
try:
    baseline_results = trainer.evaluate()
    
    end_time = time.time()
    eval_time = end_time - start_time
    
    print(f"\nEvaluation completed in {eval_time:.1f} seconds")
    print("\nBASELINE (Pre-Training) Results:")
    print(f"Accuracy: {baseline_results['eval_accuracy']:.4f} ({baseline_results['eval_accuracy']*100:.2f}%)")
    print(f"Loss: {baseline_results['eval_loss']:.4f}")
    
    print(f"\nExpected baseline: ~50% (random guessing)")
    print(f"After fine-tuning: Should reach 85-90%+ accuracy")
    print("Baseline established - ready for training!")
    
except Exception as e:
    print(f"Baseline evaluation issue: {e}")
    print("Proceeding to training anyway...")



=== Task 4: Baseline Performance Evaluation ===
🧪 Testing UNTRAINED model performance...
📊 Evaluating on 872 validation samples...
⏰ This should be fast on local device...



⏱️ Evaluation completed in 7.5 seconds

📊 BASELINE (Pre-Training) Results:
Accuracy: 0.5092 (50.92%)
Loss: 0.7475

🎯 Expected baseline: ~50% (random guessing)
🚀 After fine-tuning: Should reach 85-90%+ accuracy
✅ Baseline established - ready for training!


In [ ]:
# Cell 9: Full Parameter Fine-Tuning (Local)
print("=== Task 5: Full Parameter Fine-Tuning ===")

print("Starting BERT-base full fine-tuning on SST-2...")
print("Expected time: 5-15 minutes for 3 epochs (local)")
print("Training progress will appear below...")

# Start training
try:
    start_training = time.time()
    training_results = trainer.train()
    end_training = time.time()
    
    total_time = end_training - start_training
    
    print(f"\nTraining completed successfully in {total_time/60:.1f} minutes!")
    print("Training Results:")
    print(f"Final training loss: {training_results.training_loss:.4f}")
    
    # Evaluate trained model
    print("\nEvaluating TRAINED model...")
    final_results = trainer.evaluate()
    
    print("\nFINAL Results:")
    print(f"Accuracy: {final_results['eval_accuracy']:.4f} ({final_results['eval_accuracy']*100:.2f}%)")
    print(f"Loss: {final_results['eval_loss']:.4f}")
    
    # Calculate improvement
    if 'baseline_results' in locals():
        improvement = final_results['eval_accuracy'] - baseline_results['eval_accuracy']
        print(f"Improvement: +{improvement:.4f} ({improvement*100:.2f} percentage points)")
    
    print(f"\nTarget: 85-90%+ accuracy")
    
    # Save model with organized structure (recommended approach)
    print("\nSaving trained model...")
    import os
    os.makedirs("./models", exist_ok=True)
    model_save_path = "./models/bert-sst2-finetuned"
    
    model.save_pretrained(model_save_path)
    tokenizer.save_pretrained(model_save_path)
    
    print(f"Model saved to {model_save_path}/")
 

except Exception as e:
    print(f"Training failed: {e}")
    print("Check memory usage and batch size settings")
    
    # Memory cleanup
    torch.mps.empty_cache() if device.type == "mps" else None

=== Task 5: Full Parameter Fine-Tuning ===
🚀 Starting BERT-base full fine-tuning on SST-2...
Expected time: 5-15 minutes for 3 epochs (local)
📊 Training progress will appear below...


Step,Training Loss,Validation Loss


❌ Training failed: MPS backend out of memory (MPS allocated: 4.22 GB, other allocations: 4.81 GB, max allowed: 9.07 GB). Tried to allocate 89.42 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).
Check memory usage and batch size settings


In [ ]:
#Cell 10
print("\n=== Testing Trained Model ===")

# Test with some example sentences
test_sentences = [
    "This movie is absolutely fantastic and amazing!",
    "I really hate this boring and terrible film.",
    "The acting was decent but the plot was confusing.",
    "Best movie I've ever seen in my life!",
    "Worst waste of time and money ever."
]

print("Testing trained model on sample sentences:")

for sentence in test_sentences:
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(probabilities, dim=-1)
        confidence = probabilities.max().item()
    
    sentiment = "POSITIVE" if predicted_class.item() == 1 else "NEGATIVE"
    print(f"'{sentence[:50]}...' → {sentiment} ({confidence:.3f})")




=== Testing Trained Model ===
🧪 Testing trained model on sample sentences:
'This movie is absolutely fantastic and amazing!...' → POSITIVE (1.000)
'I really hate this boring and terrible film....' → NEGATIVE (1.000)
'The acting was decent but the plot was confusing....' → NEGATIVE (1.000)
'Best movie I've ever seen in my life!...' → POSITIVE (1.000)
'Worst waste of time and money ever....' → NEGATIVE (1.000)

🎉 Local fine-tuning complete! Your MacBook M2 successfully trained BERT! 🍎
